# ds004752 V5.6 Colab Tranche 1 Full-Cohort Signal Audit

This notebook is an orchestration layer only. All project logic stays in `src/cli.py` and package modules so runs are testable and reproducible.

Use this notebook to mount Google Drive, sync the repo, materialize the ds004752 payloads, run a full-cohort Gate 0 signal audit, inspect the resulting artifacts, and confirm real-data phases remain guarded.


## 1. Mount Drive

Artifacts, cache, checkpoints, and optional materialized data live under `/content/drive/MyDrive/eeg-ds004752`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or update the project repo

In [ ]:
# Clone hoặc pull repo private một cách an toàn.
# - Nếu repo chưa tồn tại: hỏi GitHub token bằng getpass, không in token ra output.
# - Nếu repo đã tồn tại: thử git pull thường trước; nếu fail do auth thì dùng token.

from pathlib import Path
import base64
import getpass
import os
import subprocess

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'


def run(cmd, cwd=None, check=True):
    print('\n$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)


def git_auth_header():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('Nhập GitHub token cho repo private: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return f'Authorization: Basic {basic}'


if not REPO_DIR.exists():
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    pull_result = subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
    if pull_result.returncode != 0:
        extra_header = git_auth_header()
        run(['git', '-c', f'http.extraHeader={extra_header}', 'pull', '--ff-only'], cwd=REPO_DIR)

In [ ]:
%cd /content/eeg-ds004752
!INSTALL_SIGNAL_EXTRAS=1 bash bootstrap/install_runtime.sh
!python bootstrap/colab_quickstart.py
!git log --oneline -1


## 3. Materialize payloads and run full-cohort Gate 0 signal audit

The execution cell first repairs common Colab apt dependency issues (`netbase`, broken dpkg state), materializes the full dataset payload, and then runs Gate 0 with `--include-signal` across the full cohort.


In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y netbase
!sudo apt-get install -f -y
!bash bootstrap/get_data_colab.sh /content/drive/MyDrive/eeg-ds004752/data all
!python -m src.cli audit --profile t4_safe --config configs/data/snapshot_colab.yaml --include-signal --signal-max-sessions 68


## 4. Inspect representative payload files

Check that both EEG/iEEG and beamforming files are no longer tiny git-annex pointers after full materialization.


In [ ]:
!ls -lh /content/drive/MyDrive/eeg-ds004752/data/ds004752/sub-01/ses-01/eeg
!ls -lh /content/drive/MyDrive/eeg-ds004752/data/ds004752/sub-14/ses-08/ieeg
!ls -lh /content/drive/MyDrive/eeg-ds004752/data/ds004752/derivatives/sub-14/beamforming


## 5. Run tests and smoke check


In [ ]:
!python -m unittest discover -s tests
!python -m src.cli smoke --profile t4_safe --config configs/data/snapshot_colab.yaml

## 6. Locate the latest Gate 0 run


In [ ]:
from pathlib import Path

latest = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate0/latest.txt')
run_dir = Path(latest.read_text().strip())
print(run_dir)


## 7. Inspect manifest, cohort lock, and audit summary


In [ ]:
import json
from pathlib import Path

latest = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate0/latest.txt')
run_dir = Path(latest.read_text().strip())
manifest = json.loads((run_dir / 'manifest.json').read_text())
cohort_lock = json.loads((run_dir / 'cohort_lock.json').read_text())
materialization_report = json.loads((run_dir / 'materialization_report.json').read_text())
audit_report = (run_dir / 'audit_report.md').read_text()

summary = {
    'run_dir': str(run_dir),
    'manifest_status': manifest['manifest_status'],
    'primary_eligibility_status': manifest['participants']['primary_eligibility_status'],
    'bridge_availability_status': manifest['derivatives']['bridge_availability_status'],
    'gate0_blockers': manifest['gate0_blockers'],
    'signal_status': manifest['signal_audit']['status'],
    'sessions_checked': manifest['signal_audit'].get('sessions_checked'),
    'mat_files_checked': manifest['signal_audit'].get('mat_files_checked'),
    'cohort_lock_status': cohort_lock['cohort_lock_status'],
    'n_primary_eligible': cohort_lock['n_primary_eligible'],
    'edf_materialized': materialization_report['payloads']['edf']['materialized_count'],
    'edf_total': materialization_report['payloads']['edf']['count'],
    'mat_materialized': materialization_report['payloads']['mat']['materialized_count'],
    'mat_total': materialization_report['payloads']['mat']['count'],
}

print(summary)
print('\n=== audit_report.md ===\n')
print(audit_report[:4000])


## 8. Guard check

This command is expected to stay blocked until Gate 2.5 preregistration is locked. The cell is wrapped so `Run all` can complete while still confirming the guard holds.


In [ ]:
import subprocess

cmd = [
    'python', '-m', 'src.cli', 'phase1_real',
    '--profile', 'a100_fast',
    '--config', 'configs/prereg/prereg_bundle.json',
]
result = subprocess.run(cmd, capture_output=True, text=True)
print('returncode:', result.returncode)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    raise RuntimeError('Guard unexpectedly opened; inspect prereg bundle state.')

print('Expected: real-data phase remains guarded.')
